<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module3/m3_l2_nb1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Module 3 · Lesson 2 · Notebook 1
# Building a Basic Autoencoder on MNIST

---

**Module 3 · Introduction to Autoencoders**  
**Estimated time:** 25 minutes  
**Level:** Beginner — assumes you have completed the Lesson 1 reading and concept check

---

## 📋 What This Notebook Does

In Lesson 1 you built a clear conceptual picture of what an autoencoder is and why it works. This notebook makes that concrete. You will implement the encoder–bottleneck–decoder architecture from scratch, train it on the MNIST handwritten digits dataset, and look at what it produces.

By the end of this notebook you will have:

- Loaded and normalised MNIST image data for use in a neural network
- Built an encoder and decoder as separate Keras models and connected them into a full autoencoder
- Trained the model using binary cross-entropy loss and the Adam optimiser
- Plotted training and validation loss curves and interpreted what they tell you
- Produced a side-by-side grid of original versus reconstructed digit images
- Identified which digits the model reconstructs best and worst — and thought about why

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Import libraries, fix random seeds |
| **1. Load MNIST** | Download, inspect, and understand the dataset |
| **2. Preprocess** | Flatten, normalise, verify |
| **3. Build the Encoder** | Define the compression half of the network |
| **4. Build the Decoder** | Define the reconstruction half |
| **5. Assemble and Compile** | Connect encoder + decoder, choose loss function |
| **6. Train** | Fit the model, plot loss curves |
| **7. Inspect Reconstructions** | Visualise original vs reconstructed images |
| **8. Reflect** | Guided questions about what you observed |

---

> **Before you start:** Complete the Lesson 1 reading pages and the Lesson 1 concept check. This notebook assumes you understand what an encoder, latent space, and decoder are, what the bottleneck is for, and why binary cross-entropy requires a sigmoid output activation.

---
## Step 0 — Setup

Run the cell below to import everything you need. All libraries are pre-installed on Google Colab — no additional installation required.

> **Reproducibility:** We fix the random seeds for NumPy and TensorFlow so that your results are consistent across runs. Neural network training involves randomness (weight initialisation, batch shuffling), and fixing seeds ensures you get the same output each time you run the notebook from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print('✅ All imports successful.')

---
## Step 1 — Load and Explore MNIST

### 1.1 — About the Dataset

MNIST contains **70,000 greyscale images** of handwritten digits (0–9), each 28×28 pixels. It is split into 60,000 training images and 10,000 test images. It has become the standard entry-point dataset for image-based deep learning — small enough to train quickly, structured enough to produce interpretable results.

**Why is MNIST ideal for learning autoencoders?**
- Images are tiny (784 pixels), so a CPU is sufficient — no GPU needed
- Digits are visually distinct, so you can judge reconstruction quality by eye
- There are exactly 10 classes, making the latent space scatter plot (Notebook 2) easy to interpret
- The digits have enough internal structure that the bottleneck is forced to learn something real

> **Note on labels:** You will not use the digit labels (0–9) during training — autoencoders are unsupervised. You will use them later, in Notebook 2, to colour-code the latent space scatter plot.

In [ ]:
# Load MNIST — TensorFlow includes it as a built-in dataset
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

print('Dataset loaded successfully.')
print(f'  Training images: {x_train_raw.shape}  → (samples, height, width)')
print(f'  Test images:     {x_test_raw.shape}')
print(f'  Pixel value range: {x_train_raw.min()} to {x_train_raw.max()}')
print(f'  Digit classes:   {np.unique(y_train)}')

In [ ]:
# Visualise a sample of training images — one per digit class
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))

for digit in range(10):
    # Find first occurrence of each digit
    idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(x_train_raw[idx], cmap='gray')
    axes[0, digit].set_title(str(digit), fontsize=10)
    axes[0, digit].axis('off')
    # Second example of same digit
    idx2 = np.where(y_train == digit)[0][1]
    axes[1, digit].imshow(x_train_raw[idx2], cmap='gray')
    axes[1, digit].axis('off')

plt.suptitle('Sample MNIST Training Images — Two Examples per Digit Class',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()
print('These are the images your autoencoder will learn to compress and reconstruct.')
print('Notice how much variation there is within each digit class — different handwriting styles,'
      ' stroke thicknesses, and angles.')

---
## Step 2 — Preprocess the Data

Before feeding images into the network, two preprocessing steps are required:

### 2.1 — Flatten: 28×28 → 784

Each image arrives as a 28×28 two-dimensional array. A fully-connected (dense) network expects a one-dimensional input vector. We flatten each image into 784 values. No information is lost — this is purely a reshaping operation.

### 2.2 — Normalise: [0, 255] → [0, 1]

Raw pixel values are integers from 0 (black) to 255 (white). We divide by 255 to map them to the range [0, 1]. This matters for two reasons:

1. **Training stability** — gradient descent works better with small, bounded inputs close to zero
2. **Loss function compatibility** — binary cross-entropy requires both predictions and targets to be in [0, 1]; without normalisation the loss calculation breaks

In [ ]:
# Flatten 28×28 → 784 and normalise to [0, 1]
x_train = x_train_raw.astype('float32') / 255.0   # normalise
x_test  = x_test_raw.astype('float32')  / 255.0

x_train = x_train.reshape(-1, 784)                # flatten
x_test  = x_test.reshape(-1, 784)

print('After preprocessing:')
print(f'  x_train shape: {x_train.shape}  (60,000 images × 784 pixels each)')
print(f'  x_test  shape: {x_test.shape}')
print(f'  Pixel range:   {x_train.min():.1f} to {x_train.max():.1f}  ✅ (normalised)')
print()
print('The 28×28 grid has been flattened into a single row of 784 numbers.')
print('Both input and target will be x_train — the autoencoder reconstructs its own input.')

---
## Step 3 — Build the Encoder

The encoder compresses the 784-dimensional input into a 32-dimensional latent vector. We build it as a **separate Keras model** — this allows us to use the encoder on its own later (in Notebook 2) to extract latent representations without passing through the decoder.

**Architecture:**
```
Input (784)  →  Dense(128, ReLU)  →  Dense(32)  ← bottleneck
```

**Why no activation on the bottleneck layer?**  
Leaving the bottleneck linear (no activation) allows latent values to be both positive and negative. This gives the latent space more geometric freedom, which produces better clustering in the scatter plot you will make in Notebook 2. Adding a ReLU would clip negative values to zero, artificially constraining the latent geometry.

**`LATENT_DIM`** is defined as a variable at the top so you can easily change it and re-run the experiment.

In [ ]:
# ── Hyperparameter: change this to experiment with different bottleneck sizes ──
LATENT_DIM = 32

# ── Encoder ──
encoder_input = keras.Input(shape=(784,), name='encoder_input')
x = layers.Dense(128, activation='relu', name='encoder_hidden')(encoder_input)
latent = layers.Dense(LATENT_DIM, activation=None, name='latent_space')(x)  # no activation

encoder = Model(encoder_input, latent, name='encoder')

print(f'Encoder built — bottleneck: {LATENT_DIM} dimensions')
print(f'Compression ratio: {LATENT_DIM}/784 = {LATENT_DIM/784*100:.1f}% of original size')
print()
encoder.summary()

---
## Step 4 — Build the Decoder

The decoder is the mirror image of the encoder. It takes the 32-dimensional latent vector and expands it back to 784 dimensions.

**Architecture:**
```
Latent (32)  →  Dense(128, ReLU)  →  Dense(784, sigmoid)
```

**Why sigmoid on the output?**  
The sigmoid function maps any real number to the range (0, 1). We need this because:
- Our normalised pixel values live in [0, 1]
- Binary cross-entropy loss requires predictions in (0, 1) — without sigmoid, the loss calculation is mathematically undefined

This is the rule from Lesson 1: **BCE loss requires sigmoid on the output layer.**

In [ ]:
# ── Decoder ──
decoder_input  = keras.Input(shape=(LATENT_DIM,), name='decoder_input')
x = layers.Dense(128, activation='relu', name='decoder_hidden')(decoder_input)
decoder_output = layers.Dense(784, activation='sigmoid', name='decoder_output')(x)  # sigmoid!

decoder = Model(decoder_input, decoder_output, name='decoder')

print('Decoder built — expands latent code back to 784 dimensions')
print('Note: sigmoid on the final layer constrains outputs to (0, 1) — required for BCE loss')
print()
decoder.summary()

---
## Step 5 — Assemble the Full Autoencoder and Compile

We connect the encoder and decoder into a single end-to-end model. During training, the full autoencoder is used — the input flows through the encoder, through the bottleneck, and through the decoder to produce a reconstruction.

After training:
- Use **`encoder`** alone to extract latent codes (Notebook 2)
- Use **`decoder`** alone to reconstruct from arbitrary latent vectors
- Use **`autoencoder`** end-to-end to go from image → image

**Loss function:** `binary_crossentropy` — treats each pixel as an independent probability and penalises confident wrong predictions via the logarithm. Produces sharper reconstructions than MSE for near-binary image data like MNIST.

**Optimiser:** `adam` with default learning rate 0.001 — the reliable starting point for most neural network training tasks.

In [ ]:
# ── Full Autoencoder ──
autoencoder_input = keras.Input(shape=(784,), name='autoencoder_input')
encoded = encoder(autoencoder_input)
decoded = decoder(encoded)

autoencoder = Model(autoencoder_input, decoded, name='autoencoder')

autoencoder.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

print('Full autoencoder assembled and compiled.')
print()
autoencoder.summary()
print()
print(f'Total trainable parameters: {autoencoder.count_params():,}')
print(f'Each of these will be updated by gradient descent during training.')

### 5.1 — Verify the Parameter Count

Let's manually verify the parameter count for the first encoder layer to confirm we understand what `summary()` is reporting.

**Encoder hidden layer:**  
- Inputs: 784  
- Neurons: 128  
- Parameters = (784 × 128 weights) + 128 biases = **100,480**

This is the number of individual numbers gradient descent will adjust in that one layer alone.

In [ ]:
# Verify parameter count for first encoder layer
w, b = encoder.get_layer('encoder_hidden').get_weights()
print(f'Encoder hidden layer:')
print(f'  Weight matrix shape: {w.shape}  (784 inputs × 128 neurons)')
print(f'  Bias vector shape:   {b.shape}  (one bias per neuron)')
print(f'  Total parameters:    {w.size + b.size:,}')
print(f'  Formula:             ({w.shape[0]} × {w.shape[1]}) + {b.shape[0]} = {w.size + b.size:,}')

---
## Step 6 — Train the Autoencoder

### 6.1 — What the Training Call Does

The key detail in the training call below:

```python
autoencoder.fit(x_train, x_train, ...)   # input = target = the same images
```

Both the input **and** the target are `x_train`. The network is penalised whenever its output differs from its input. This is the self-supervised training signal that requires no labels.

**`validation_split=0.1`** reserves 10% of training data as a validation set. The model never trains on these images, but we compute the loss on them after each epoch to monitor generalisation.

**What to watch:** Both curves should decrease together and plateau at similar values. A large growing gap between them would indicate overfitting — but for a simple autoencoder on MNIST this is unlikely.

In [ ]:
print('Training the autoencoder...')
print('Input = Target = x_train  (the network reconstructs its own input)')
print()

history = autoencoder.fit(
    x_train, x_train,           # input AND target are the same
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.1,       # 10% held out for validation monitoring
    verbose=1
)

print()
print('✅ Training complete.')

### 6.2 — Plot the Loss Curves

**What a healthy training run looks like:**
- Both curves decrease steadily in the first few epochs
- Both plateau at similar values — a small gap is normal
- No sharp divergence between training and validation

**Common warning signs:**
- Loss not decreasing at all → check normalisation and loss/activation pairing
- Loss oscillating wildly → learning rate may be too high
- Large and growing gap → overfitting (unlikely here, but worth knowing)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(history.history['loss'],
        label='Training loss', color='steelblue', linewidth=2)
ax.plot(history.history['val_loss'],
        label='Validation loss', color='tomato', linewidth=2, linestyle='--')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=12)
ax.set_title(f'Autoencoder Training Curves  (LATENT_DIM = {LATENT_DIM})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

final_train = history.history['loss'][-1]
final_val   = history.history['val_loss'][-1]
print(f'Final training loss:   {final_train:.5f}')
print(f'Final validation loss: {final_val:.5f}')
print(f'Gap (train - val):     {final_train - final_val:.5f}')

---
## Step 7 — Inspect the Reconstructions

### 7.1 — Original vs Reconstructed Grid

We pass test images through the full autoencoder and display them side by side.

**What to look for:**
- Are the reconstructions recognisably the same digit as the originals?
- How blurry are they? Blurriness is expected — the bottleneck discards fine-grained pixel detail, and the decoder fills in smooth averages
- Which digits are reconstructed most faithfully? Which are most distorted?

> **Some blurriness is not a failure — it is evidence that compression is happening.** A pixel-perfect reconstruction would mean the bottleneck is large enough that no real compression is occurring.

In [ ]:
# Generate reconstructions on the test set
reconstructed = autoencoder.predict(x_test, verbose=0)

n_display = 10
fig, axes = plt.subplots(2, n_display, figsize=(16, 3.8))

for i in range(n_display):
    # Top row: originals
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Original\n({y_test[i]})', fontsize=7)
    axes[0, i].axis('off')
    # Bottom row: reconstructions
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title('Recon', fontsize=7)
    axes[1, i].axis('off')

plt.suptitle(f'Original (top) vs Reconstructed (bottom)  —  LATENT_DIM = {LATENT_DIM}',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### 7.2 — Best and Worst Reconstructions

We compute the **mean squared error (MSE)** between each test image and its reconstruction as a per-image quality score. Lower MSE = better reconstruction.

Looking at the best and worst cases tells us something concrete about what the autoencoder has learned — and what it struggles with.

In [ ]:
# Per-image reconstruction error
mse_per_image = np.mean((x_test - reconstructed) ** 2, axis=1)

print(f'Reconstruction MSE across test set:')
print(f'  Mean:   {mse_per_image.mean():.5f}')
print(f'  Lowest: {mse_per_image.min():.5f}  (best reconstruction)')
print(f'  Highest:{mse_per_image.max():.5f}  (worst reconstruction)')

# Show best and worst
best_idx  = np.argmin(mse_per_image)
worst_idx = np.argmax(mse_per_image)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))

for col, (idx, label) in enumerate([
    (best_idx,  'Best — MSE = {:.5f}'.format(mse_per_image[best_idx])),
    (worst_idx, 'Worst — MSE = {:.5f}'.format(mse_per_image[worst_idx]))
]):
    axes[0, col*2].imshow(x_test[idx].reshape(28,28), cmap='gray')
    axes[0, col*2].set_title(f'{label}\nDigit: {y_test[idx]}\nOriginal', fontsize=8)
    axes[0, col*2].axis('off')
    axes[1, col*2].imshow(reconstructed[idx].reshape(28,28), cmap='gray')
    axes[1, col*2].set_title('Reconstructed', fontsize=8)
    axes[1, col*2].axis('off')
    # Difference image
    diff = np.abs(x_test[idx] - reconstructed[idx])
    axes[0, col*2+1].imshow(diff.reshape(28,28), cmap='Reds', vmin=0, vmax=0.5)
    axes[0, col*2+1].set_title('Error map\n(red = large error)', fontsize=8)
    axes[0, col*2+1].axis('off')
    axes[1, col*2+1].axis('off')

plt.suptitle('Best and Worst Reconstructions with Error Maps', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# Per-class average reconstruction error
print('\nAverage reconstruction MSE per digit class:')
for digit in range(10):
    mask      = y_test == digit
    class_mse = mse_per_image[mask].mean()
    bar = '█' * int(class_mse * 10000)
    print(f'  Digit {digit}: {class_mse:.5f}  {bar}')

---
## Step 8 — Guided Reflection

Answer the questions below by replacing the placeholder text. Write in your own words — these are thinking prompts, not right-or-wrong quiz questions.

---

### ❓ Question 1
**Look at the reconstruction grid. Which digits are reconstructed most clearly and which are most blurry or distorted? Does the per-class error table (Step 7.2) confirm your visual impression?**

*Think about: why might simple digits like 1 reconstruct better than complex digits like 4 or 8?*

> **Your answer:** *(Replace this text)*

---

### ❓ Question 2
**Look at the loss curves. Do the training and validation curves plateau at similar values, or is there a visible gap? What does this tell you about whether the model is overfitting?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 3
**The reconstructions are blurrier than the originals. Is this a problem? What does it tell you about what the bottleneck has learned to encode?**

*Think about: if the reconstruction were pixel-perfect, what would that imply about the bottleneck?*

> **Your answer:** *(Replace this text)*

---

### ❓ Question 4 — Optional experiment
**Change `LATENT_DIM` at the top of Step 3 from 32 to 8, then re-run all cells from Step 3 onwards. How do the reconstructions change? How does the final loss change? Does the gap between best and worst reconstruction MSE grow or shrink?**

> **Your answer:** *(Replace this text)*

---
## ✅ Notebook 1 Complete

You have:

| ✅ | Achievement |
|----|-------------|
| ✅ | Loaded and explored MNIST — 70,000 handwritten digit images |
| ✅ | Preprocessed: flattened to 784D and normalised to [0, 1] |
| ✅ | Built encoder (784 → 128 → 32) and decoder (32 → 128 → 784) as separate Keras models |
| ✅ | Assembled the full autoencoder and compiled with BCE loss |
| ✅ | Trained for 20 epochs and interpreted the loss curves |
| ✅ | Produced a reconstruction grid and identified best/worst performing digit classes |

---

### ➡️ Next Step

Read the **Notebook 2: Exploring the Latent Space** reading page before opening Notebook 2. That page explains what the 2D scatter plot should look like and what to look for when you see it.

> **Keep this notebook session open.** Notebook 2 will reuse the `encoder` and `decoder` objects you just trained.